# 다이닝코드 v1 재실험: MF · DeepCoNN · Hybrid

기준: [`diningcode_analysis.ipynb`](./diningcode_analysis.ipynb). 수정 목록: [`REVISION_PLAN.md`](./REVISION_PLAN.md) (아래 `[x-y]`는 그 문서의 항목 번호).

세 모델을 **같은 데이터 · 같은 split · 같은 선택 규칙**으로 비교한다.

| 단계 | 사용 데이터 | 하는 일 |
|---|---|---|
| 선택 | inner-train → validation | 하이퍼파라미터 · epoch 선택 (test 미사용) |
| 최종 | train 전체 (= inner + validation) | 선택된 설정으로 seed 3개 재학습, test 1회 평가 |

- 데이터: `crawled_data/` 서울 · 경기 (원본 notebook과 동일)
- 결과물: `results/` (`summary.md`, `metrics.json`, `predictions_test.csv`, `run_config.json`)
- 실행: 이 폴더에서 `.venv` 커널로 전체 실행. CPU 기준 약 1.5시간

In [1]:
import os, sys, json, time, random, pickle, hashlib, platform
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder

HERE = Path.cwd()
assert (HERE / 'crawled_data').exists(), 'v1_rating_prediction 폴더에서 실행해야 한다'

# ---- 데이터 [2-4] : 원본 notebook과 같은 서울·경기 파일 ----
SOURCE_FILES = [
    'crawled_data/diningcode_data_crawling_seoul_20260125_1542.csv',
    'crawled_data/diningcode_data_crawling_gyeonggi_20260322_1244.csv',
]

# ---- 경로 [1-4] : Windows 절대 경로 대신 환경 변수 / 폴더 상대 경로 ----
CACHE_DIR = HERE / '.cache'
CACHE_DIR.mkdir(exist_ok=True)
FASTTEXT_PATH = Path(os.environ.get('FASTTEXT_PATH', CACHE_DIR / 'cc.ko.300.vec.gz'))
if 'JAVA_HOME' not in os.environ and (HERE / '.venv/lib/jvm').exists():
    os.environ['JAVA_HOME'] = str(HERE / '.venv/lib/jvm')

# REVISION_SMOKE=1 이면 배치 수·epoch을 줄여 파이프라인만 점검한다 (결과는 임시 폴더)
SMOKE = os.environ.get('REVISION_SMOKE') == '1'
RESULTS_DIR = Path('/tmp/revision_smoke_results') if SMOKE else HERE / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- 공통 설정 ----
SPLIT_SEED = 42
SEEDS = [42] if SMOKE else [42, 43, 44]          # [1-5]
MAX_VOCAB_SIZE = 20000
MAX_DOC_LENGTH = 550
EMBED_DIM = 300

torch.set_num_threads(os.cpu_count())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SPLIT_SEED)                                # [1-2] 맨 위에서 고정
print(f'device={DEVICE}, threads={torch.get_num_threads()}, smoke={SMOKE}, results -> {RESULTS_DIR}')

device=cpu, threads=12, smoke=False, results -> /home/hananthony1/codes/ai-ds-projects/projects/rating_recsys/legacy/v1_rating_prediction/results


## 1. 데이터 전처리

유지 [0]: 다코미식가 분리, 평점 파싱, `'...더보기\n'` 제거, `drop_duplicates()`.
추가 [2-2]: 중복 제거 후 남는 같은 (사용자, 식당) 쌍은 최초 1건만 남긴다.
삭제 [2-3]: 맛 · 가격 · 응대 인코딩. 정답 리뷰 작성자가 남긴 값이라 입력으로 쓰지 않는다.

In [2]:
raw = pd.concat([pd.read_csv(HERE / f) for f in SOURCE_FILES], ignore_index=True)
data = raw.drop(columns=['Unnamed: 0'])

data['daco_gourmand'] = data['user_name'].str.contains('다코미식가', regex=False).astype(int)
data['user_name'] = (data['user_name'].str.replace('다코미식가', '', regex=False)
                     .str.replace('\r\n', '', regex=False).str.replace('\n', '', regex=False))
data['user_rating'] = data['user_rating'].str.replace('점', '', regex=False).astype(float)
data['user_query'] = data['user_query'].str.replace('...더보기\n', '', regex=False)

n_raw = len(data)
data = data.drop_duplicates()
n_dedup = len(data)
n_dup_pairs = int(data.duplicated(['user_name', 'item_name']).sum())
data = data.drop_duplicates(['user_name', 'item_name'], keep='first')

data['user_id'] = LabelEncoder().fit_transform(data['user_name'])
data['item_id'] = LabelEncoder().fit_transform(data['item_name'])
data = data.rename(columns={'user_rating': 'rating', 'user_query': 'review'})
N_USERS, N_ITEMS = data['user_id'].nunique(), data['item_id'].nunique()

data_stats = {'raw_rows': n_raw, 'after_drop_duplicates': n_dedup,
              'duplicate_user_item_pairs_removed': n_dup_pairs, 'final_rows': len(data),
              'users': N_USERS, 'items': N_ITEMS,
              'sparsity': round(1 - len(data) / (N_USERS * N_ITEMS), 4)}
data_stats

{'raw_rows': 19297,
 'after_drop_duplicates': 13944,
 'duplicate_user_item_pairs_removed': 10,
 'final_rows': 13934,
 'users': 5205,
 'items': 353,
 'sparsity': 0.9924}

## 2. Split (세 모델 공통) [1-1, 2-1, 6-3]

- 사용자별 80/20 (`random_state=42`)은 원본과 같다. 리뷰 1건 사용자는 전부 train에 남아 test에 나오지 않는다 → 결과는 "리뷰 2건 이상 사용자" 기준.
- Train에서 train 이력이 2건 이상인 사용자마다 1건을 validation으로 뗀다.
- Train에 없는 식당의 test 행은 MF가 예측할 수 없으므로 세 모델 모두에서 제외한다.

In [3]:
train = data.groupby('user_id').sample(frac=0.8, random_state=SPLIT_SEED)
test = data.drop(train.index)

cnt = train.groupby('user_id').size()
val = (train[train['user_id'].isin(cnt[cnt >= 2].index)]
       .groupby('user_id').sample(n=1, random_state=SPLIT_SEED))
inner = train.drop(val.index)

val = val[val['item_id'].isin(inner['item_id']) & val['user_id'].isin(inner['user_id'])]
test_eval = test[test['item_id'].isin(train['item_id']) & test['user_id'].isin(train['user_id'])]

# train/test 행이 겹치지 않는지 확인
assert not set(train.index) & set(test.index)
assert not set(inner.index) & set(val.index)

split_stats = {'train': len(train), 'inner_train': len(inner), 'validation': len(val),
               'test': len(test), 'test_evaluated': len(test_eval),
               'test_users': int(test_eval['user_id'].nunique()),
               'single_review_users_not_tested': int((data.groupby('user_id').size() == 1).sum())}
split_stats

{'train': 11841,
 'inner_train': 9449,
 'validation': 2392,
 'test': 2093,
 'test_evaluated': 2093,
 'test_users': 1557,
 'single_review_users_not_tested': 2813}

## 3. 평가 함수 [6-1, 6-4, 6-5]

- 주 지표: RMSE, MAE. 모든 모델의 예측을 [1, 5]로 clip (실제 평점이 이 범위 안이라 오차가 늘지 않는다).
- P@3 · R@3 · NDCG@3은 쓰지 않는다 [6-2]. 사용자별 held-out 식당만 정렬하고 전부 relevant로 두어 어떤 모델이든 같은 값(0.4301 / 0.9901)이 나오고, NDCG@3도 무작위 순서가 약 0.986이다.
- 모델 간 차이: 같은 test 행에서 paired bootstrap 95% 구간.

In [4]:
def clip(p):
    return np.clip(np.asarray(p, dtype=float), 1.0, 5.0)

def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(p))))

def paired_bootstrap_diff(y, preds_a, preds_b, n_boot=2000, seed=0):
    # preds_*: (n_seeds, n_rows). 통계량 = seed 평균 RMSE(a) - seed 평균 RMSE(b)
    y = np.asarray(y)
    ea, eb = (np.asarray(preds_a) - y) ** 2, (np.asarray(preds_b) - y) ** 2
    stat = lambda idx: np.sqrt(ea[:, idx].mean(1)).mean() - np.sqrt(eb[:, idx].mean(1)).mean()
    rng = np.random.default_rng(seed)
    boots = np.array([stat(rng.integers(0, len(y), len(y))) for _ in range(n_boot)])
    return float(stat(np.arange(len(y)))), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

y_val = val['rating'].to_numpy()
y_test = test_eval['rating'].to_numpy()

# 모델별 기록: validation 선택 결과, seed별 test 예측
REPORT = {}   # name -> {'val_rmse':..., 'params':..., 'test_preds': {seed: array}}

## 4. 참고 baseline [3-5]

- 전체 평균: train 평균
- User + item bias: `μ + b_u + b_i`, 정규화 강도를 validation으로 선택 후 train 전체로 재적합

In [5]:
def fit_bias(frame, reg_u, reg_i, iters=15):
    mu = frame['rating'].mean()
    b_u = pd.Series(0.0, index=frame['user_id'].unique())
    for _ in range(iters):
        g = (frame['rating'] - mu - frame['user_id'].map(b_u)).groupby(frame['item_id'])
        b_i = g.sum() / (g.count() + reg_i)
        g = (frame['rating'] - mu - frame['item_id'].map(b_i)).groupby(frame['user_id'])
        b_u = g.sum() / (g.count() + reg_u)
    return lambda f: clip(mu + f['user_id'].map(b_u).fillna(0).to_numpy()
                          + f['item_id'].map(b_i).fillna(0).to_numpy())

mu_inner = inner['rating'].mean()
REPORT['Global mean'] = {'val_rmse': rmse(y_val, np.full(len(y_val), mu_inner)), 'params': {},
                         'test_preds': {s: np.full(len(y_test), train['rating'].mean()) for s in SEEDS}}

bias_grid = {(ru, ri): rmse(y_val, fit_bias(inner, ru, ri)(val))
             for ru in [1, 3, 5, 10, 20] for ri in [1, 3, 5, 10, 25, 50]}
best_bias = min(bias_grid, key=bias_grid.get)
bias_pred = fit_bias(train, *best_bias)(test_eval)
REPORT['User+item bias'] = {'val_rmse': bias_grid[best_bias],
                            'params': {'reg_user': best_bias[0], 'reg_item': best_bias[1]},
                            'test_preds': {s: bias_pred for s in SEEDS}}
{k: round(v['val_rmse'], 4) for k, v in REPORT.items()}, best_bias

({'Global mean': 0.7246, 'User+item bias': 0.6769}, (3, 25))

## 5. Matrix Factorization [3-1 ~ 3-4]

`Study/RecSys/matrixfactorization/matfac.py`는 다른 실험도 쓰므로 수정하지 않고, 같은 SGD update 규칙의 class를 여기에 둔다.

| 원본 | 수정 |
|---|---|
| P · Q ~ U[0, 6/k) → 초기 예측 약 5.3점 | N(0, 0.1) |
| μ = 식당별 평균의 평균 (4.447) | train 전체 평균 |
| Loss에 관측되지 않은 칸까지 합산 | 관측값만으로 train / validation RMSE |
| k=10, lr 0.001, reg 0.02, 50 epoch 고정 | validation으로 선택 |
| 행 순서 고정, seed 없음 | epoch마다 seed 고정 셔플 |

In [6]:
class MF:
    # r_hat = mu + b_u + b_i + p_u . q_i, matfac.py와 같은 순차 SGD update
    def __init__(self, frame, k, lr, reg, seed):
        self.k, self.lr, self.reg = k, lr, reg
        self.rng = np.random.default_rng(seed)
        self.u = frame['user_id'].to_numpy()
        self.i = frame['item_id'].to_numpy()
        self.r = frame['rating'].to_numpy(dtype=float)
        self.mu = float(self.r.mean())
        self.P = self.rng.normal(0, 0.1, (N_USERS, k))
        self.Q = self.rng.normal(0, 0.1, (N_ITEMS, k))
        self.bu, self.bi = np.zeros(N_USERS), np.zeros(N_ITEMS)

    def run_epoch(self):
        P, Q, bu, bi, lr, reg, mu = self.P, self.Q, self.bu, self.bi, self.lr, self.reg, self.mu
        for t in self.rng.permutation(len(self.r)):
            u, i = self.u[t], self.i[t]
            e = self.r[t] - (mu + bu[u] + bi[i] + P[u] @ Q[i])
            P[u] += lr * (e * Q[i] - reg * P[u])
            Q[i] += lr * (e * P[u] - reg * Q[i])
            bu[u] += lr * (e - reg * bu[u])
            bi[i] += lr * (e - reg * bi[i])

    def predict(self, frame, clip_output=True):
        u, i = frame['user_id'].to_numpy(), frame['item_id'].to_numpy()
        p = self.mu + self.bu[u] + self.bi[i] + np.einsum('nk,nk->n', self.P[u], self.Q[i])
        return clip(p) if clip_output else p

    def train_rmse(self):
        p = self.mu + self.bu[self.u] + self.bi[self.i] + np.einsum('nk,nk->n', self.P[self.u], self.Q[self.i])
        return rmse(self.r, p)

In [7]:
MF_MAX_EPOCHS = 5 if SMOKE else 100
MF_GRID = ([(10, 0.005, 0.1)] if SMOKE else
           [(k, lr, reg) for k in [10, 32] for lr in [0.002, 0.005, 0.01] for reg in [0.02, 0.1]])

t0 = time.time()
mf_curves, mf_best = {}, None
for k, lr, reg in MF_GRID:
    m = MF(inner, k, lr, reg, seed=SPLIT_SEED)
    curve = []
    for ep in range(1, MF_MAX_EPOCHS + 1):
        m.run_epoch()
        curve.append(rmse(y_val, m.predict(val)))
    ep_best = int(np.argmin(curve)) + 1
    mf_curves[f'k{k}_lr{lr}_reg{reg}'] = [round(v, 4) for v in curve]
    print(f'k={k:2d} lr={lr:<6} reg={reg:<5} best epoch {ep_best:3d}  val RMSE {min(curve):.4f}  '
          f'(train RMSE {m.train_rmse():.4f} @ {MF_MAX_EPOCHS})')
    if mf_best is None or min(curve) < mf_best['val_rmse']:
        mf_best = {'k': k, 'lr': lr, 'reg': reg, 'epochs': ep_best, 'val_rmse': min(curve)}
print(f'selected: {mf_best}  ({time.time() - t0:.0f}s)')

k=10 lr=0.002  reg=0.02  best epoch 100  val RMSE 0.6859  (train RMSE 0.5868 @ 100)


k=10 lr=0.002  reg=0.1   best epoch 100  val RMSE 0.6828  (train RMSE 0.6105 @ 100)


k=10 lr=0.005  reg=0.02  best epoch  43  val RMSE 0.6856  (train RMSE 0.3537 @ 100)


k=10 lr=0.005  reg=0.1   best epoch  59  val RMSE 0.6806  (train RMSE 0.4752 @ 100)


k=10 lr=0.01   reg=0.02  best epoch  23  val RMSE 0.6856  (train RMSE 0.1743 @ 100)


...

k=32 lr=0.002  reg=0.1   best epoch 100  val RMSE 0.6863  (train RMSE 0.5760 @ 100)


k=32 lr=0.005  reg=0.02  best epoch  35  val RMSE 0.6934  (train RMSE 0.2359 @ 100)


k=32 lr=0.005  reg=0.1   best epoch  63  val RMSE 0.6848  (train RMSE 0.4036 @ 100)


k=32 lr=0.01   reg=0.02  best epoch  17  val RMSE 0.6935  (train RMSE 0.0969 @ 100)


k=32 lr=0.01   reg=0.1   best epoch  29  val RMSE 0.6850  (train RMSE 0.2715 @ 100)
selected: {'k': 10, 'lr': 0.01, 'reg': 0.1, 'epochs': 28, 'val_rmse': 0.6805928619108766}  (135s)


In [8]:
# 최종: train 전체로 seed별 재학습, 선택된 epoch 수만큼
mf_models, mf_test_preds = {}, {}
for s in SEEDS:
    m = MF(train, mf_best['k'], mf_best['lr'], mf_best['reg'], seed=s)
    for ep in range(1, mf_best['epochs'] + 1):
        m.run_epoch()
        if ep % 10 == 0 or ep == mf_best['epochs']:
            print(f'seed {s} epoch {ep:3d}  train RMSE (관측값) {m.train_rmse():.4f}')
    mf_models[s], mf_test_preds[s] = m, m.predict(test_eval)
    print(f'seed {s}: test RMSE {rmse(y_test, mf_test_preds[s]):.4f}')

REPORT['MF'] = {'val_rmse': mf_best['val_rmse'],
                'params': {k: mf_best[k] for k in ['k', 'lr', 'reg', 'epochs']},
                'test_preds': mf_test_preds}

seed 42 epoch  10  train RMSE (관측값) 0.6569


seed 42 epoch  20  train RMSE (관측값) 0.6009


seed 42 epoch  28  train RMSE (관측값) 0.5652
seed 42: test RMSE 0.6807


seed 43 epoch  10  train RMSE (관측값) 0.6570


seed 43 epoch  20  train RMSE (관측값) 0.6007


seed 43 epoch  28  train RMSE (관측값) 0.5649
seed 43: test RMSE 0.6804


seed 44 epoch  10  train RMSE (관측값) 0.6566


seed 44 epoch  20  train RMSE (관측값) 0.6006


seed 44 epoch  28  train RMSE (관측값) 0.5648
seed 44: test RMSE 0.6810


## 6. 텍스트 입력 [4-1 ~ 4-3]

원본은 test 문서를 **test 리뷰로** 만들고, train 문서에 **각 행의 정답 리뷰**가 들어 있었다. 수정:

- 리뷰 단위로 Okt(`stem=True`) 토큰화를 한 번 하고 캐시한다.
- 문서 = 해당 사용자(식당)의 **학습 이력 리뷰**를 원래 행 순서대로 이어 붙인 뒤 550 토큰에서 자른다.
- 학습 행: 그 행의 리뷰를 뺀 문서 (leave-one-out). Validation · test 행: 학습 이력 전체.
- Vocabulary도 학습 이력 리뷰로만 만든다 (상위 20,000).

In [9]:
from konlpy.tag import Okt

texts = data['review'].astype(str)
tok_key = hashlib.sha256('\x1e'.join(texts).encode('utf-8')).hexdigest()[:16]
tok_path = CACHE_DIR / f'okt_review_tokens_{tok_key}.pkl'
if tok_path.exists():
    TOKENS = pickle.loads(tok_path.read_bytes())
    print(f'loaded token cache {tok_path.name}')
else:
    okt, t0 = Okt(), time.time()
    TOKENS = {idx: okt.morphs(t, stem=True) for idx, t in texts.items()}
    tok_path.write_bytes(pickle.dumps(TOKENS))
    print(f'tokenized {len(TOKENS)} reviews in {time.time() - t0:.0f}s')

lens = pd.Series({k: len(v) for k, v in TOKENS.items()})
lens.describe().round(1)

loaded token cache okt_review_tokens_3f9521732a4471f0.pkl


count    13934.0
mean        37.8
std         40.8
min          0.0
25%         16.0
50%         24.0
75%         42.0
max        413.0
dtype: float64

In [10]:
def build_vocab(history):
    counts = Counter(t for idx in history.index for t in TOKENS[idx])
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for w, _ in counts.most_common(MAX_VOCAB_SIZE):
        vocab[w] = len(vocab)
    return vocab

def build_docs(history, rows, vocab, key, leave_one_out):
    # rows 각 행의 key(user_id / item_id) 문서. history 행 순서대로 이어 붙이고 MAX_DOC_LENGTH에서 자른다.
    groups = {k: sorted(v) for k, v in history.groupby(key).groups.items()}
    enc = {idx: [vocab.get(t, 1) for t in TOKENS[idx]] for idx in history.index}
    out = np.zeros((len(rows), MAX_DOC_LENGTH), dtype=np.int64)
    n_empty = 0
    for n, (ridx, k) in enumerate(zip(rows.index, rows[key])):
        seq = []
        for h in groups.get(k, []):
            if leave_one_out and h == ridx:
                continue
            seq.extend(enc[h])
            if len(seq) >= MAX_DOC_LENGTH:
                break
        seq = seq[:MAX_DOC_LENGTH]
        out[n, :len(seq)] = seq
        n_empty += len(seq) == 0
    return out, n_empty

def unk_ratio(docs):
    nz = docs[docs != 0]
    return float((nz == 1).mean()) if nz.size else 0.0

def prepare_stage(history, eval_rows):
    # history: 학습 행(= 문서 이력), eval_rows: 평가 행
    vocab = build_vocab(history)
    stage = {'vocab': vocab, 'fit_rows': history, 'eval_rows': eval_rows}
    stage['fit_u'], e1 = build_docs(history, history, vocab, 'user_id', leave_one_out=True)
    stage['fit_i'], e2 = build_docs(history, history, vocab, 'item_id', leave_one_out=True)
    stage['eval_u'], e3 = build_docs(history, eval_rows, vocab, 'user_id', leave_one_out=False)
    stage['eval_i'], e4 = build_docs(history, eval_rows, vocab, 'item_id', leave_one_out=False)
    stage['stats'] = {'vocab_size': len(vocab),
                      'empty_user_docs_fit': int(e1), 'empty_item_docs_fit': int(e2),
                      'empty_user_docs_eval': int(e3), 'empty_item_docs_eval': int(e4),
                      'unk_ratio_eval_user': round(unk_ratio(stage['eval_u']), 4),
                      'unk_ratio_eval_item': round(unk_ratio(stage['eval_i']), 4)}
    return stage

t0 = time.time()
STAGE_SELECT = prepare_stage(inner, val)       # 선택 단계
STAGE_FINAL = prepare_stage(train, test_eval)  # 최종 단계
print(f'docs built in {time.time() - t0:.0f}s')
pd.DataFrame({'select (inner→val)': STAGE_SELECT['stats'], 'final (train→test)': STAGE_FINAL['stats']})

docs built in 2s


,select (inner→val),final (train→test)
vocab_size,12394.0,13550.0
empty_user_docs_fit,4179.0,2966.0
empty_item_docs_fit,0.0,0.0
empty_user_docs_eval,60.0,20.0
empty_item_docs_eval,0.0,0.0
unk_ratio_eval_user,0.0,0.0
unk_ratio_eval_item,0.0,0.0


빈 문서: 학습 이력이 1건뿐인 사용자의 학습 행은 leave-one-out 문서가 비어 PAD만 남는다. 개수는 위 표에 기록했다.

fastText `cc.ko.300` (상위 300,000 단어)로 embedding을 초기화한다. OOV는 원본처럼 N(0, 0.6), PAD는 0.

In [11]:
from gensim.models import KeyedVectors

t0 = time.time()
FASTTEXT = KeyedVectors.load_word2vec_format(str(FASTTEXT_PATH), binary=False, limit=300000)
print(f'fastText loaded in {time.time() - t0:.0f}s')

def build_embedding(vocab, seed):
    rng = np.random.default_rng(seed)
    mat = np.zeros((len(vocab), EMBED_DIM), dtype=np.float32)
    covered = 0
    for w, idx in vocab.items():
        if w == '<PAD>':
            continue
        if w in FASTTEXT:
            mat[idx] = FASTTEXT[w]
            covered += 1
        else:
            mat[idx] = rng.normal(scale=0.6, size=EMBED_DIM)
    return mat, covered / (len(vocab) - 1)

for name, st in [('select', STAGE_SELECT), ('final', STAGE_FINAL)]:
    _, cov = build_embedding(st['vocab'], 0)
    st['stats']['fasttext_coverage'] = round(cov, 4)
    print(f'{name}: vocab {len(st["vocab"])}, fastText coverage {cov:.1%}')

fastText loaded in 42s
select: vocab 12394, fastText coverage 63.6%
final: vocab 13550, fastText coverage 62.4%


## 7. DeepCoNN · Hybrid 모델 [4-4, 4-5, 5]

한 class로 두 모델을 만든다.

```text
DeepCoNN : r̂ = g + FM([x_u, y_i])
Hybrid   : r̂ = g + FM([x_u, y_i]) + b_u + b_i + p_u·q_i
```

- `x_u`, `y_i`: 사용자 · 식당 문서 CNN(커널 3, 100 filter, max-pool) → FC(32). 원본 구조 유지.
- 원본 대비 수정
  - Dropout을 FM 2차항이 아니라 CNN 출력(FC 입력)에 적용 [4-5]
  - `fm_V`를 U[0, 1) 대신 주석에 적힌 의도대로 U[-0.1, 0.1] 초기화. 원본의 1 epoch 평균 loss 282,131 발산 원인 중 하나
  - Global bias를 0 대신 train 평균으로 초기화
- Hybrid의 ID 부분(`b_u, b_i, p_u, q_i`)은 MF에서 선택된 k와 reg를 쓰고, reg는 batch L2 항으로 넣는다.
- 학습: MSE, RMSprop(alpha 0.9, weight decay 1e-4), batch 64, gradient clipping 1.0. lr은 validation으로 선택, 최대 20 epoch, early stopping patience 3 [4-4].

In [12]:
class TextRatingModel(nn.Module):
    def __init__(self, embedding_matrix, global_mean, use_id=False, id_k=10,
                 num_filters=100, kernel_size=3, latent_dim=32, fm_k=16, dropout=0.3):
        super().__init__()
        self.use_id = use_id
        self.embedding = nn.Embedding.from_pretrained(torch.as_tensor(embedding_matrix), freeze=False, padding_idx=0)
        E = embedding_matrix.shape[1]
        self.user_cnn = nn.Conv1d(E, num_filters, kernel_size, padding=kernel_size // 2)
        self.item_cnn = nn.Conv1d(E, num_filters, kernel_size, padding=kernel_size // 2)
        self.user_fc = nn.Linear(num_filters, latent_dim)
        self.item_fc = nn.Linear(num_filters, latent_dim)
        self.dropout = nn.Dropout(dropout)
        self.fm_linear = nn.Linear(2 * latent_dim, 1)
        self.fm_V = nn.Parameter(torch.empty(2 * latent_dim, fm_k).uniform_(-0.1, 0.1))
        self.global_bias = nn.Parameter(torch.tensor([float(global_mean)]))
        if use_id:
            self.user_bias = nn.Embedding(N_USERS, 1)
            self.item_bias = nn.Embedding(N_ITEMS, 1)
            self.P = nn.Embedding(N_USERS, id_k)
            self.Q = nn.Embedding(N_ITEMS, id_k)
            for emb in (self.user_bias, self.item_bias):
                nn.init.zeros_(emb.weight)
            for emb in (self.P, self.Q):
                nn.init.normal_(emb.weight, 0, 0.1)

    def tower(self, doc, cnn, fc):
        h = F.relu(cnn(self.embedding(doc).transpose(1, 2)))   # (B, filters, L)
        h = h.max(dim=2).values                                  # max-pool
        return fc(self.dropout(h))

    def forward(self, u_doc, i_doc, u_idx, i_idx, return_parts=False):
        z = F.relu(torch.cat([self.tower(u_doc, self.user_cnn, self.user_fc),
                              self.tower(i_doc, self.item_cnn, self.item_fc)], dim=1))
        linear = self.fm_linear(z).squeeze(1)
        xv = z @ self.fm_V
        quadratic = 0.5 * (xv ** 2 - (z ** 2) @ (self.fm_V ** 2)).sum(1)
        parts = {'global_bias': self.global_bias.expand_as(linear), 'text_linear': linear, 'text_quadratic': quadratic}
        if self.use_id:
            parts['id_bias'] = (self.user_bias(u_idx) + self.item_bias(i_idx)).squeeze(1)
            parts['id_interaction'] = (self.P(u_idx) * self.Q(i_idx)).sum(1)
        pred = sum(parts.values())
        return (pred, parts) if return_parts else pred

    def id_l2(self, u_idx, i_idx):
        if not self.use_id:
            return 0.0
        return (self.P(u_idx).pow(2).sum(1) + self.Q(i_idx).pow(2).sum(1)
                + self.user_bias(u_idx).pow(2).squeeze(1) + self.item_bias(i_idx).pow(2).squeeze(1)).mean()

In [13]:
BATCH_SIZE = 64
DC_MAX_EPOCHS = 1 if SMOKE else 20
DC_PATIENCE = 3
DC_LR_GRID = [1e-3] if SMOKE else [1e-3, 3e-4]
MAX_BATCHES = 20 if SMOKE else None     # smoke 모드에서만 epoch당 배치 수 제한

def make_loader(stage, part, shuffle, seed=0):
    rows = stage['fit_rows'] if part == 'fit' else stage['eval_rows']
    ds = TensorDataset(torch.as_tensor(stage[f'{part}_u']), torch.as_tensor(stage[f'{part}_i']),
                       torch.as_tensor(rows['user_id'].to_numpy()), torch.as_tensor(rows['item_id'].to_numpy()),
                       torch.as_tensor(rows['rating'].to_numpy(), dtype=torch.float32))
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=BATCH_SIZE if shuffle else 512, shuffle=shuffle, generator=g)

@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    return clip(torch.cat([model(u.to(DEVICE), i.to(DEVICE), ui.to(DEVICE), ii.to(DEVICE)).cpu()
                           for u, i, ui, ii, _ in loader]).numpy())

def train_text_model(kind, stage, lr, seed, max_epochs, early_stop, log_prefix=''):
    # kind: 'DeepCoNN' | 'Hybrid'. early_stop=True 이면 eval_rows(validation)로 epoch 선택
    set_seed(seed)
    emb, _ = build_embedding(stage['vocab'], seed)
    model = TextRatingModel(emb, stage['fit_rows']['rating'].mean(), use_id=(kind == 'Hybrid'),
                            id_k=mf_best['k']).to(DEVICE)
    opt = torch.optim.RMSprop(model.parameters(), lr=lr, alpha=0.9, weight_decay=1e-4)
    fit_loader = make_loader(stage, 'fit', shuffle=True, seed=seed)
    eval_loader = make_loader(stage, 'eval', shuffle=False)
    y_eval = stage['eval_rows']['rating'].to_numpy()
    history, best = [], {'val_rmse': np.inf, 'epoch': 0, 'state': None}
    for ep in range(1, max_epochs + 1):
        model.train()
        t0, se, n = time.time(), 0.0, 0
        for b, (u, i, ui, ii, y) in enumerate(fit_loader):
            if MAX_BATCHES and b >= MAX_BATCHES:
                break
            u, i, ui, ii, y = u.to(DEVICE), i.to(DEVICE), ui.to(DEVICE), ii.to(DEVICE), y.to(DEVICE)
            pred = model(u, i, ui, ii)
            mse = F.mse_loss(pred, y)
            loss = mse + mf_best['reg'] * model.id_l2(ui, ii)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            se, n = se + mse.item() * len(y), n + len(y)
        rec = {'epoch': ep, 'train_rmse': round(float(np.sqrt(se / n)), 4), 'seconds': round(time.time() - t0, 1)}
        if early_stop:
            rec['val_rmse'] = round(rmse(y_eval, predict_model(model, eval_loader)), 4)
            if rec['val_rmse'] < best['val_rmse']:
                best = {'val_rmse': rec['val_rmse'], 'epoch': ep,
                        'state': {k: v.detach().clone() for k, v in model.state_dict().items()}}
        history.append(rec)
        print(log_prefix, rec, flush=True)
        if early_stop and ep - best['epoch'] >= DC_PATIENCE:
            break
    if early_stop:
        model.load_state_dict(best['state'])
    return model, history, best['epoch'], best['val_rmse']

### 7-1. Validation으로 lr · epoch 선택 (seed 42)

In [14]:
TEXT_SELECTION = {}
for kind in ['DeepCoNN', 'Hybrid']:
    runs = []
    for lr in DC_LR_GRID:
        _, hist, ep, v = train_text_model(kind, STAGE_SELECT, lr, SPLIT_SEED, DC_MAX_EPOCHS,
                                          early_stop=True, log_prefix=f'[{kind} lr={lr}]')
        runs.append({'lr': lr, 'epochs': ep, 'val_rmse': v, 'history': hist})
    best = min(runs, key=lambda r: r['val_rmse'])
    TEXT_SELECTION[kind] = {'best': {k: best[k] for k in ['lr', 'epochs', 'val_rmse']}, 'runs': runs}
    print(f'>>> {kind} selected: {TEXT_SELECTION[kind]["best"]}')

[DeepCoNN lr=0.001] {'epoch': 1, 'train_rmse': 0.7875, 'seconds': 57.3, 'val_rmse': 0.7164}


[DeepCoNN lr=0.001] {'epoch': 2, 'train_rmse': 0.7743, 'seconds': 48.4, 'val_rmse': 0.7096}


[DeepCoNN lr=0.001] {'epoch': 3, 'train_rmse': 0.7647, 'seconds': 47.7, 'val_rmse': 0.7032}


[DeepCoNN lr=0.001] {'epoch': 4, 'train_rmse': 0.7562, 'seconds': 47.5, 'val_rmse': 0.7025}


[DeepCoNN lr=0.001] {'epoch': 5, 'train_rmse': 0.7497, 'seconds': 48.1, 'val_rmse': 0.7086}


...

[Hybrid lr=0.0003] {'epoch': 4, 'train_rmse': 0.7489, 'seconds': 49.4, 'val_rmse': 0.6985}


[Hybrid lr=0.0003] {'epoch': 5, 'train_rmse': 0.7391, 'seconds': 54.0, 'val_rmse': 0.7014}


[Hybrid lr=0.0003] {'epoch': 6, 'train_rmse': 0.7328, 'seconds': 53.6, 'val_rmse': 0.7056}


[Hybrid lr=0.0003] {'epoch': 7, 'train_rmse': 0.7309, 'seconds': 50.7, 'val_rmse': 0.6991}


>>> Hybrid selected: {'lr': 0.0003, 'epochs': 4, 'val_rmse': 0.6985}


### 7-2. 최종 학습 (train 전체, seed 3개) → test 1회 평가

In [15]:
TEXT_MODELS = {}
for kind in ['DeepCoNN', 'Hybrid']:
    sel = TEXT_SELECTION[kind]['best']
    preds, TEXT_MODELS[kind] = {}, {}
    for s in SEEDS:
        model, hist, _, _ = train_text_model(kind, STAGE_FINAL, sel['lr'], s, sel['epochs'],
                                             early_stop=False, log_prefix=f'[{kind} final seed={s}]')
        preds[s] = predict_model(model, make_loader(STAGE_FINAL, 'eval', shuffle=False))
        TEXT_MODELS[kind][s] = model
        print(f'{kind} seed {s}: test RMSE {rmse(y_test, preds[s]):.4f}', flush=True)
    REPORT[kind] = {'val_rmse': sel['val_rmse'],
                    'params': {'lr': sel['lr'], 'epochs': sel['epochs'],
                               **({'id_k': mf_best['k'], 'id_reg': mf_best['reg']} if kind == 'Hybrid' else {})},
                    'test_preds': preds}

[DeepCoNN final seed=42] {'epoch': 1, 'train_rmse': 0.7724, 'seconds': 61.6}


[DeepCoNN final seed=42] {'epoch': 2, 'train_rmse': 0.7566, 'seconds': 61.1}


[DeepCoNN final seed=42] {'epoch': 3, 'train_rmse': 0.7449, 'seconds': 63.1}


[DeepCoNN final seed=42] {'epoch': 4, 'train_rmse': 0.7352, 'seconds': 62.7}


DeepCoNN seed 42: test RMSE 0.6964


...

[Hybrid final seed=44] {'epoch': 1, 'train_rmse': 0.7732, 'seconds': 69.5}


[Hybrid final seed=44] {'epoch': 2, 'train_rmse': 0.7555, 'seconds': 67.2}


[Hybrid final seed=44] {'epoch': 3, 'train_rmse': 0.7421, 'seconds': 67.4}


[Hybrid final seed=44] {'epoch': 4, 'train_rmse': 0.731, 'seconds': 67.3}


Hybrid seed 44: test RMSE 0.6958


## 8. 결과 비교 [6-1, 6-5, 7]

In [16]:
MODEL_ORDER = ['Global mean', 'User+item bias', 'MF', 'DeepCoNN', 'Hybrid']
mf_mat = np.stack([REPORT['MF']['test_preds'][s] for s in SEEDS])

rows = []
for name in MODEL_ORDER:
    mat = np.stack([REPORT[name]['test_preds'][s] for s in SEEDS])
    r = [rmse(y_test, p) for p in mat]
    a = [mae(y_test, p) for p in mat]
    row = {'model': name, 'val_rmse': REPORT[name]['val_rmse'],
           'test_rmse_mean': np.mean(r), 'test_rmse_sd': np.std(r, ddof=1) if len(r) > 1 else 0.0,
           'test_mae_mean': np.mean(a), 'test_mae_sd': np.std(a, ddof=1) if len(a) > 1 else 0.0}
    if name != 'MF':
        d, lo, hi = paired_bootstrap_diff(y_test, mat, mf_mat)
        row.update({'diff_vs_mf': d, 'diff_ci_low': lo, 'diff_ci_high': hi})
    rows.append(row)
results_df = pd.DataFrame(rows).set_index('model')
results_df.round(4)

,val_rmse,test_rmse_mean,test_rmse_sd,test_mae_mean,test_mae_sd,diff_vs_mf,diff_ci_low,diff_ci_high
model,,,,,,,,
Global mean,0.7246,0.7422,0.0000,0.6230,0.0000,0.0615,0.0469,0.0765
User+item bias,0.6769,0.6772,0.0000,0.5244,0.0000,-0.0035,-0.0068,-0.0002
MF,0.6806,0.6807,0.0003,0.5172,0.0002,NaN,NaN,NaN
DeepCoNN,0.6987,0.6949,0.0021,0.5509,0.0033,0.0142,0.0053,0.0230
Hybrid,0.6985,0.6946,0.0016,0.5502,0.0044,0.0139,0.0053,0.0222


In [17]:
# 사용자 train 이력 수별 RMSE (seed 평균 예측 기준) [6-5]
hist_cnt = test_eval['user_id'].map(train.groupby('user_id').size())
bucket = pd.cut(hist_cnt, bins=[0, 2, 9, np.inf], labels=['1-2', '3-9', '10+'])
bucket_rows = {}
for name in MODEL_ORDER:
    p = np.mean([REPORT[name]['test_preds'][s] for s in SEEDS], axis=0)
    bucket_rows[name] = {b: rmse(y_test[(bucket == b).to_numpy()], p[(bucket == b).to_numpy()])
                         for b in bucket.cat.categories}
bucket_df = pd.DataFrame(bucket_rows).T
bucket_df.loc['n_rows'] = bucket.value_counts().reindex(bucket.cat.categories).to_numpy()
bucket_df.round(4)

,1-2,3-9,10+
Global mean,0.7468,0.7361,0.7526
User+item bias,0.6986,0.6789,0.6493
MF,0.6967,0.6838,0.6528
DeepCoNN,0.7118,0.6954,0.6615
Hybrid,0.7120,0.6947,0.6610
n_rows,483.0000,1151.0000,459.0000


## 9. 해석 [4-6]

Hybrid 예측을 구성 항목으로 나눈다 (`forward(..., return_parts=True)`, 모델 내부 값). 텍스트 부분과 ID 부분이 각각 평균에서 얼마나 움직이는지 본다.

In [18]:
@torch.no_grad()
def decompose(model, stage):
    model.eval()
    out = []
    for u, i, ui, ii, _ in make_loader(stage, 'eval', shuffle=False):
        _, parts = model(u.to(DEVICE), i.to(DEVICE), ui.to(DEVICE), ii.to(DEVICE), return_parts=True)
        out.append(pd.DataFrame({k: v.cpu().numpy() for k, v in parts.items()}))
    return pd.concat(out, ignore_index=True)

parts_df = decompose(TEXT_MODELS['Hybrid'][SEEDS[0]], STAGE_FINAL)
parts_df['text_total'] = parts_df['text_linear'] + parts_df['text_quadratic']
parts_df['id_total'] = parts_df['id_bias'] + parts_df['id_interaction']
parts_df[['global_bias', 'text_total', 'id_total', 'id_bias', 'id_interaction']].describe().round(4)

,global_bias,text_total,id_total,id_bias,id_interaction
count,2093.0000,2093.0000,2093.0000,2093.0000,2093.0000
mean,4.3921,-0.0509,0.0076,0.0076,-0.0000
std,0.0000,0.2402,0.0210,0.0176,0.0115
min,4.3921,-1.0508,-0.0730,-0.0499,-0.0516
25%,4.3921,-0.2018,-0.0057,-0.0044,-0.0052
50%,4.3921,-0.0390,0.0075,0.0069,0.0002
75%,4.3921,0.1231,0.0208,0.0193,0.0059
max,4.3921,0.5860,0.0879,0.0750,0.0448


In [19]:
# 추천 예시: test 행이 가장 많은 사용자, 모델별 예측(seed 평균)과 실제 평점
names = data.set_index('user_id')['user_name'].to_dict()
ex_user = test_eval['user_id'].value_counts().idxmax()
mask = (test_eval['user_id'] == ex_user).to_numpy()
example = test_eval.loc[mask, ['item_name', 'rating']].copy()
for name in ['MF', 'DeepCoNN', 'Hybrid']:
    example[name] = np.mean([REPORT[name]['test_preds'][s] for s in SEEDS], axis=0)[mask]
print(f"user '{names[ex_user]}' (train 이력 {int(train['user_id'].eq(ex_user).sum())}건)")
example.sort_values('Hybrid', ascending=False).round(3)

user 'ronny' (train 이력 42건)


,item_name,rating,MF,DeepCoNN,Hybrid
14074,포폴로피자,5.0,5.000,4.945,4.922
14397,호운,5.0,5.000,4.942,4.904
15045,꼬꼬오리주물럭,5.0,5.000,4.949,4.880
14907,정통 춘천 닭갈비,5.0,5.000,4.929,4.871
3538,정돈,4.5,4.905,4.907,4.842
16864,코울러,5.0,4.930,4.870,4.840
12704,강민주의 들밥 본점,5.0,4.933,4.904,4.828
2506,명동교자 분점,5.0,4.868,4.847,4.788
11168,호우섬,5.0,4.626,4.670,4.575
15806,능라도 분당점,5.0,4.323,4.595,4.475


## 10. 결과 저장 [6-6]

In [20]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

# seed별 test 예측
pred_out = test_eval[['user_id', 'item_id', 'user_name', 'item_name', 'rating']].copy()
for name in MODEL_ORDER:
    if name in ('Global mean', 'User+item bias'):
        pred_out[name] = REPORT[name]['test_preds'][SEEDS[0]]
    else:
        for s in SEEDS:
            pred_out[f'{name}_seed{s}'] = REPORT[name]['test_preds'][s]
pred_out.round(5).to_csv(RESULTS_DIR / 'predictions_test.csv', index_label='row_index', encoding='utf-8-sig')

metrics = {
    'data': data_stats, 'split': split_stats,
    'text_stages': {'select': STAGE_SELECT['stats'], 'final': STAGE_FINAL['stats']},
    'comparison': json.loads(results_df.round(5).reset_index().to_json(orient='records', force_ascii=False)),
    'rmse_by_user_history': json.loads(bucket_df.round(5).to_json(force_ascii=False)),
    'selection': {'User+item bias': REPORT['User+item bias']['params'], 'MF': mf_best,
                  'MF_curves_val_rmse': mf_curves,
                  **{k: v for k, v in TEXT_SELECTION.items()}},
}
(RESULTS_DIR / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2, default=float))

run_config = {
    'notebook': 'diningcode_revision.ipynb', 'smoke': SMOKE,
    'source_files': {f: sha256(HERE / f) for f in SOURCE_FILES},
    'fasttext': {'path': FASTTEXT_PATH.name, 'limit': 300000},
    'split_seed': SPLIT_SEED, 'model_seeds': SEEDS,
    'text': {'max_vocab_size': MAX_VOCAB_SIZE, 'max_doc_length': MAX_DOC_LENGTH, 'batch_size': BATCH_SIZE,
             'max_epochs': DC_MAX_EPOCHS, 'patience': DC_PATIENCE, 'lr_grid': DC_LR_GRID,
             'optimizer': 'RMSprop(alpha=0.9, weight_decay=1e-4)', 'grad_clip': 1.0, 'dropout': 0.3,
             'num_filters': 100, 'kernel_size': 3, 'latent_dim': 32, 'fm_k': 16},
    'mf': {'grid': MF_GRID, 'max_epochs': MF_MAX_EPOCHS, 'init': 'N(0, 0.1)'},
    'selected': {name: REPORT[name]['params'] for name in MODEL_ORDER},
    'env': {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__,
            'pandas': pd.__version__, 'device': str(DEVICE), 'threads': torch.get_num_threads()},
}
(RESULTS_DIR / 'run_config.json').write_text(json.dumps(run_config, ensure_ascii=False, indent=2, default=str))

2149

In [21]:
# results/summary.md 생성
def fmt_row(name, r):
    diff = '–' if name == 'MF' else f"{r.diff_vs_mf:+.4f} [{r.diff_ci_low:+.4f}, {r.diff_ci_high:+.4f}]"
    sd = lambda v: f' ± {v:.4f}' if name in ('MF', 'DeepCoNN', 'Hybrid') else ''
    return (f"| {name} | {r.val_rmse:.4f} | {r.test_rmse_mean:.4f}{sd(r.test_rmse_sd)} | "
            f"{r.test_mae_mean:.4f}{sd(r.test_mae_sd)} | {diff} |")

p = {n: REPORT[n]['params'] for n in MODEL_ORDER}
lines = [
    '# v1 재실험 결과: MF · DeepCoNN · Hybrid', '',
    '`diningcode_revision.ipynb` 실행 시 자동 생성된다. 수정 내용은 [`../REVISION_PLAN.md`](../REVISION_PLAN.md).', '',
    '## 조건', '',
    f"- 데이터: 서울 · 경기 크롤링 {data_stats['raw_rows']:,}행 → 중복 제거 {data_stats['final_rows']:,}행 "
    f"(사용자 {data_stats['users']:,}명, 식당 {data_stats['items']}곳, sparsity {data_stats['sparsity']:.2%})",
    f"- Split: 사용자별 80/20 (seed {SPLIT_SEED}). Train {split_stats['train']:,} = inner {split_stats['inner_train']:,} "
    f"+ validation {split_stats['validation']:,}. Test {split_stats['test_evaluated']:,}행 / {split_stats['test_users']:,}명 "
    f"(리뷰 2건 이상 사용자, train에 있는 식당)",
    '- 선택은 validation으로만, test는 train 전체로 재학습한 뒤 1회 평가. 모든 예측은 [1, 5] clip',
    f"- 모델 seed {SEEDS}: 평균 ± 표준편차. 차이는 MF 대비 RMSE, paired bootstrap 95% 구간 (음수 = MF보다 낮음)",
    '- 텍스트 문서: 학습 이력 리뷰로만 구성, 학습 행은 자기 리뷰 제외 (leave-one-out). 맛 · 가격 · 응대 미사용', '',
    '## 비교', '',
    '| 모델 | Validation RMSE | Test RMSE | Test MAE | vs MF ΔRMSE [95% CI] |',
    '|---|---:|---:|---:|---:|',
    *[fmt_row(n, results_df.loc[n]) for n in MODEL_ORDER], '',
    '선택된 설정', '',
    f"- User+item bias: reg_user {p['User+item bias']['reg_user']}, reg_item {p['User+item bias']['reg_item']}",
    f"- MF: k {p['MF']['k']}, lr {p['MF']['lr']}, reg {p['MF']['reg']}, {p['MF']['epochs']} epochs",
    f"- DeepCoNN: lr {p['DeepCoNN']['lr']}, {p['DeepCoNN']['epochs']} epochs",
    f"- Hybrid: lr {p['Hybrid']['lr']}, {p['Hybrid']['epochs']} epochs, ID k {p['Hybrid']['id_k']}, ID reg {p['Hybrid']['id_reg']}", '',
    '## 사용자 train 이력 수별 Test RMSE (seed 평균 예측)', '',
    '| 모델 | ' + ' | '.join(bucket_df.columns) + ' |',
    '|---|' + '---:|' * len(bucket_df.columns),
    *[f"| {n} | " + ' | '.join(f'{v:.4f}' for v in bucket_df.loc[n]) + ' |' for n in MODEL_ORDER],
    '| 행 수 | ' + ' | '.join(str(int(v)) for v in bucket_df.loc['n_rows']) + ' |', '',
    '## 이전 기록과의 관계', '',
    '기존 README 수치(MF 0.7098, DeepCoNN v1 0.8045, Improved 0.5749)는 split · 입력 문서 · 선택 규칙이 달라 위 표와 비교하지 않는다. '
    'DeepCoNN v1은 test 리뷰로 test 문서를 만들었고, Improved는 정답 리뷰가 입력 문서와 side feature에 들어 있었으며 test 기준으로 checkpoint를 골랐다. '
    '원본 MF 설정 점검 결과는 `v1_mf_audit_metrics.json`에 있다 (고정 설정 test RMSE 0.7088, validation 튜닝 시 0.6848, user+item bias 0.6745).', '',
    '## 파일', '',
    '| 파일 | 내용 |', '|---|---|',
    '| `summary.md` | 이 문서 |',
    '| `metrics.json` | 데이터 · split 통계, 비교표, 이력 구간별 RMSE, 선택 과정 (MF grid 곡선, DeepCoNN · Hybrid epoch별 기록) |',
    '| `predictions_test.csv` | Test 행별 실제 평점과 모델 · seed별 예측 |',
    '| `run_config.json` | 입력 파일 SHA-256, seed, 하이퍼파라미터, 실행 환경 |',
    '| `v1_mf_audit_metrics.json` | 원본 MF 측정 점검 결과 (matfac 동작, 튜닝 부족, baseline) |',
]
(RESULTS_DIR / 'summary.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('\n'.join(lines[:30]))

# v1 재실험 결과: MF · DeepCoNN · Hybrid

`diningcode_revision.ipynb` 실행 시 자동 생성된다. 수정 내용은 [`../REVISION_PLAN.md`](../REVISION_PLAN.md).

## 조건

- 데이터: 서울 · 경기 크롤링 19,297행 → 중복 제거 13,934행 (사용자 5,205명, 식당 353곳, sparsity 99.24%)
- Split: 사용자별 80/20 (seed 42). Train 11,841 = inner 9,449 + validation 2,392. Test 2,093행 / 1,557명 (리뷰 2건 이상 사용자, train에 있는 식당)
- 선택은 validation으로만, test는 train 전체로 재학습한 뒤 1회 평가. 모든 예측은 [1, 5] clip
- 모델 seed [42, 43, 44]: 평균 ± 표준편차. 차이는 MF 대비 RMSE, paired bootstrap 95% 구간 (음수 = MF보다 낮음)
- 텍스트 문서: 학습 이력 리뷰로만 구성, 학습 행은 자기 리뷰 제외 (leave-one-out). 맛 · 가격 · 응대 미사용

## 비교

| 모델 | Validation RMSE | Test RMSE | Test MAE | vs MF ΔRMSE [95% CI] |
|---|---:|---:|---:|---:|
| Global mean | 0.7246 | 0.7422 | 0.6230 | +0.0615 [+0.0469, +0.0765] |
| User+item bias | 0.6769 | 0.6772 | 0.5244 | -0.0035 [-0.0068, -0.0002] |
| MF | 0.6806 | 0.6807 ± 0.0003 | 0.5172 ± 0.0002 | – |
| DeepCoNN | 0.6987 | 0.6949 ± 0.0021 | 0.5509 ± 0.0033 | +0.0142 [+0.0053, +0.0230] |
| Hybrid | 0.6985 | 0.

## 11. 정리

Test 2,093행 (리뷰 2건 이상 사용자 1,557명), seed 3개 평균 기준.

| 모델 | Test RMSE | vs MF ΔRMSE [95% CI] |
|---|---:|---:|
| User+item bias (참고) | 0.6772 | −0.0035 [−0.0068, −0.0002] |
| MF | 0.6807 ± 0.0003 | – |
| DeepCoNN | 0.6949 ± 0.0021 | +0.0142 [+0.0053, +0.0230] |
| Hybrid | 0.6946 ± 0.0016 | +0.0139 [+0.0053, +0.0222] |

- 누수를 제거하면 **DeepCoNN · Hybrid 모두 MF보다 RMSE가 높다**. 차이의 95% 구간이 0을 포함하지 않는다. 사용자 이력 구간(1–2 / 3–9 / 10+) 어디에서도 텍스트 모델이 MF보다 낮지 않다.
- 튜닝한 MF도 user+item bias baseline을 넘지 못한다. 이 데이터에서 잠재 요인 p·q가 설명하는 부분이 거의 없다.
- Hybrid의 ID 부분은 거의 학습되지 않았다 (9장: `id_total` 표준편차 0.021, 텍스트 부분 0.240). lr 3e-4 · 4 epoch 동안 ID 임베딩이 움직이지 않아 Hybrid ≈ DeepCoNN이 됐다. ID 부분의 lr을 따로 두거나 MF 학습값으로 초기화하는 것이 다음 실험 후보다.
- 기존 README의 "MF 대비 RMSE 19% 감소"는 정답 리뷰가 입력에 들어간 조건의 수치이므로, 이 조건에서는 재현되지 않는다.
